# 03 — Model Training

Trains two base models:
1. **LightGBM** — gradient boosted trees, handles class imbalance natively via `scale_pos_weight`
2. **MLP** — simple neural network, captures non-linear interactions

Both models output fraud probability. Their outputs feed the meta-learner in `04_stacking.ipynb`.

In [ ]:
!pip install -q lightgbm shap optuna

In [ ]:
import pandas as pd
import numpy as np
import lightgbm as lgb
import shap
import pickle
import json
import warnings
from sklearn.metrics import average_precision_score, roc_auc_score

warnings.filterwarnings('ignore')

train = pd.read_parquet('/kaggle/working/train_features.parquet')
val   = pd.read_parquet('/kaggle/working/val_features.parquet')

with open('/kaggle/working/feature_cols.json') as f:
    feature_cols = json.load(f)

X_train = train[feature_cols].values
y_train = train['isFraud'].values
X_val   = val[feature_cols].values
y_val   = val['isFraud'].values

fraud_rate = y_train.mean()
scale_pos_weight = (1 - fraud_rate) / fraud_rate

print(f'Train: {X_train.shape}, fraud rate: {fraud_rate:.4%}')
print(f'scale_pos_weight: {scale_pos_weight:.1f}')

## LightGBM

In [ ]:
lgb_params = {
    'objective': 'binary',
    'metric': ['auc', 'average_precision'],
    'boosting_type': 'gbdt',
    'num_leaves': 127,
    'learning_rate': 0.05,
    'feature_fraction': 0.8,
    'bagging_fraction': 0.8,
    'bagging_freq': 5,
    'scale_pos_weight': scale_pos_weight,
    'min_child_samples': 20,
    'reg_alpha': 0.1,
    'reg_lambda': 0.1,
    'random_state': 42,
    'verbose': -1,
}

dtrain = lgb.Dataset(X_train, label=y_train, feature_name=feature_cols)
dval   = lgb.Dataset(X_val, label=y_val, feature_name=feature_cols)

callbacks = [
    lgb.early_stopping(stopping_rounds=50),
    lgb.log_evaluation(period=100),
]

lgb_model = lgb.train(
    lgb_params,
    dtrain,
    num_boost_round=2000,
    valid_sets=[dval],
    callbacks=callbacks,
)

lgb_val_preds = lgb_model.predict(X_val)
print(f'\nLightGBM Val ROC-AUC: {roc_auc_score(y_val, lgb_val_preds):.4f}')
print(f'LightGBM Val PR-AUC:  {average_precision_score(y_val, lgb_val_preds):.4f}')

# Save predictions for stacking
np.save('/kaggle/working/lgbm_val_preds.npy', lgb_val_preds)
lgb_train_preds = lgb_model.predict(X_train)
np.save('/kaggle/working/lgbm_train_preds.npy', lgb_train_preds)

In [ ]:
# SHAP — feature importance
import matplotlib.pyplot as plt

explainer = shap.TreeExplainer(lgb_model)
shap_values = explainer.shap_values(X_val[:500])  # Sample for speed

plt.figure(figsize=(10, 8))
shap.summary_plot(shap_values, X_val[:500], feature_names=feature_cols,
                  plot_type='bar', show=False, max_display=20)
plt.tight_layout()
plt.savefig('/kaggle/working/shap_importance.png', dpi=150)
plt.show()

In [ ]:
# Save LightGBM model
with open('/kaggle/working/lgbm_model.pkl', 'wb') as f:
    pickle.dump(lgb_model, f)

with open('/kaggle/working/feature_names.pkl', 'wb') as f:
    pickle.dump(feature_cols, f)

print('LightGBM model saved.')

## MLP (PyTorch)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled   = scaler.transform(X_val)

with open('/kaggle/working/scaler.pkl', 'wb') as f:
    pickle.dump(scaler, f)

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Training MLP on: {device}')

X_t = torch.FloatTensor(X_train_scaled).to(device)
y_t = torch.FloatTensor(y_train).to(device)
X_v = torch.FloatTensor(X_val_scaled).to(device)
y_v = torch.FloatTensor(y_val).to(device)

train_ds = TensorDataset(X_t, y_t)
train_loader = DataLoader(train_ds, batch_size=4096, shuffle=True)


class FraudMLP(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Linear(64, 1),
            nn.Sigmoid(),
        )

    def forward(self, x):
        return self.net(x).squeeze(1)


model = FraudMLP(X_train.shape[1]).to(device)

pos_weight = torch.tensor([scale_pos_weight]).to(device)
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3, weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, patience=3)

best_val_auc = 0
best_state = None

for epoch in range(30):
    model.train()
    epoch_loss = 0
    for xb, yb in train_loader:
        optimizer.zero_grad()
        preds = model(xb)
        loss = criterion(preds, yb)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item()

    model.eval()
    with torch.no_grad():
        val_preds = model(X_v).cpu().numpy()

    val_auc = roc_auc_score(y_val, val_preds)
    val_pr  = average_precision_score(y_val, val_preds)
    scheduler.step(1 - val_auc)

    if val_auc > best_val_auc:
        best_val_auc = val_auc
        best_state = {k: v.clone() for k, v in model.state_dict().items()}

    if (epoch + 1) % 5 == 0:
        print(f'Epoch {epoch+1:2d} | loss: {epoch_loss/len(train_loader):.4f} | AUC: {val_auc:.4f} | PR-AUC: {val_pr:.4f}')

model.load_state_dict(best_state)

model.eval()
with torch.no_grad():
    mlp_val_preds  = model(X_v).cpu().numpy()
    mlp_train_preds = model(X_t).cpu().numpy()

np.save('/kaggle/working/mlp_val_preds.npy', mlp_val_preds)
np.save('/kaggle/working/mlp_train_preds.npy', mlp_train_preds)

torch.save(model.state_dict(), '/kaggle/working/mlp_state_dict.pt')
with open('/kaggle/working/mlp_model.pkl', 'wb') as f:
    pickle.dump(model, f)

print(f'\nBest MLP Val AUC: {best_val_auc:.4f}')